# Luxury Fragrance Dataset — Exploratory Analysis

Initial exploration of the luxury fragrance dataset stored as Parquet.

Objectives:

- Understand the dataset structure
- Identify the fragrance and brand fields
- Determine the number of unique brands
- Examine brand distribution
- Check data quality and missing values
- Identify potential brand-name inconsistencies

In [0]:
# Load Parquet data
from pyspark.sql import functions as F

df = spark.read.parquet(
    "/Volumes/aidc_inc_luxury_perfume_prices_full/aidcdatasetshare/full_free_2025_luxury_perfume_prices"
)

# Dataset overview
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

df.printSchema()

# Inspect columns and sample rows
print(df.columns)
display(df.limit(20))

# Check missing values
missing_values = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(missing_values)

# Explore unique brands
brand_list = (
    df.select("brand")
      .where(F.col("brand").isNotNull())
      .distinct()
      .orderBy("brand")
)

display(brand_list)

# Count unique brands
print(f"Unique brands: {brand_list.count():,}")

# Count fragrances by brand
brand_counts = (
    df.groupBy("brand")
      .count()
      .orderBy(F.col("count").desc())
)

display(brand_counts)

# Explore fragrance types
display(
    df.groupBy("type")
      .count()
      .orderBy(F.col("count").desc())
)

display(
    df.groupBy("perfumeType")
      .count()
      .orderBy(F.col("count").desc())
)

# Explore brand and fragrance details
display(
    df.select(
        "brand",
        "title",
        "type",
        "perfumeType",
        "price",
        "priceWithCurrency"
    )
    .orderBy("brand", "title")
    .limit(100)
)